In [1]:
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

users = [f"u{i:02d}" for i in range(1, 21)]
stores = ["Warszawa", "Kraków", "Gdańsk", "Wrocław"]
categories = ["elektronika", "odzież", "żywność", "książki"]

def generate_transaction():
    return {
        "tx_id": f"TX{random.randint(1, 9999):04d}",
        "user_id": random.choice(users),
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(stores),
        "category": random.choice(categories),
        "timestamp": datetime.now().isoformat()
    }

while True:
    tx = generate_transaction()
    producer.send('transactions', tx)
    print(f"Wysłano: {tx}")
    time.sleep(1)

Wysłano: {'tx_id': 'TX2592', 'user_id': 'u19', 'amount': 1110.2, 'store': 'Gdańsk', 'category': 'żywność', 'timestamp': '2026-05-07T21:28:45.468209'}
Wysłano: {'tx_id': 'TX4540', 'user_id': 'u13', 'amount': 4227.0, 'store': 'Gdańsk', 'category': 'elektronika', 'timestamp': '2026-05-07T21:28:46.476510'}
Wysłano: {'tx_id': 'TX4732', 'user_id': 'u13', 'amount': 1651.64, 'store': 'Warszawa', 'category': 'elektronika', 'timestamp': '2026-05-07T21:28:47.480267'}
Wysłano: {'tx_id': 'TX4140', 'user_id': 'u05', 'amount': 616.65, 'store': 'Gdańsk', 'category': 'żywność', 'timestamp': '2026-05-07T21:28:48.482317'}
Wysłano: {'tx_id': 'TX9656', 'user_id': 'u10', 'amount': 4645.45, 'store': 'Warszawa', 'category': 'odzież', 'timestamp': '2026-05-07T21:28:49.487801'}
Wysłano: {'tx_id': 'TX0765', 'user_id': 'u20', 'amount': 4257.32, 'store': 'Warszawa', 'category': 'książki', 'timestamp': '2026-05-07T21:28:50.491438'}
Wysłano: {'tx_id': 'TX5606', 'user_id': 'u05', 'amount': 153.22, 'store': 'Warszawa'

KeyboardInterrupt: 

In [2]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='filter-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")
print("-" * 60)

for message in consumer:
    tx = message.value
    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']} PLN | {tx['store']} | {tx['category']}")

Nasłuchuję na duże transakcje (amount > 3000)...
------------------------------------------------------------
ALERT: TX1435 | 3964.27 PLN | Wrocław | elektronika


KeyboardInterrupt: 

In [3]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Wzbogacanie transakcji o risk_level...")
print("-" * 60)

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        tx['risk_level'] = 'HIGH'
    elif tx['amount'] > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'

    print(f"{tx['tx_id']} | {tx['amount']} PLN | {tx['risk_level']}")

Wzbogacanie transakcji o risk_level...
------------------------------------------------------------
TX2354 | 721.81 PLN | LOW
TX8030 | 516.99 PLN | LOW
TX1675 | 2930.11 PLN | MEDIUM
TX5391 | 2132.84 PLN | MEDIUM
TX6669 | 1952.6 PLN | MEDIUM
TX4150 | 554.33 PLN | LOW


KeyboardInterrupt: 

In [4]:
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Zliczanie transakcji per sklep...")
print("-" * 60)

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']

Zliczanie transakcji per sklep...
------------------------------------------------------------


KeyboardInterrupt: 

In [5]:
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {'count': 0, 'total': 0.0, 'min': float('inf'), 'max': float('-inf')})
msg_count = 0

print("Statystyki per kategoria...")
print("-" * 60)

for message in consumer:
    tx = message.value
    cat = tx['category']
    amount = tx['amount']

    stats[cat]['count'] += 1
    stats[cat]['total'] += amount
    stats[cat]['min'] = min(stats[cat]['min'], amount)
    stats[cat]['max'] = max(stats[cat]['max'], amount)
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n📊 Statystyki per kategoria po {msg_count} wiadomościach:")

Statystyki per kategoria...
------------------------------------------------------------

📊 Statystyki per kategoria po 10 wiadomościach:

📊 Statystyki per kategoria po 20 wiadomościach:

📊 Statystyki per kategoria po 30 wiadomościach:


KeyboardInterrupt: 